# Gradients and Gradient Descent

## Objective
Derive gradients for scalar linear regression, compare them with autograd, and perform repeated SGD updates.

## Model and Loss

$$
\hat{y}_i = wx_i + b
$$

$$
L = \frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i-y_i)^2
$$

$$
\frac{\partial L}{\partial w}
= \frac{2}{N}\sum_i(\hat{y}_i-y_i)x_i,
\qquad
\frac{\partial L}{\partial b}
= \frac{2}{N}\sum_i(\hat{y}_i-y_i).
$$

## Test Configuration
- Inputs: `[1, 2, 3]`
- Targets: `[2, 4, 6]`
- Initial parameters: `w = 0.5`, `b = 0`
- Learning rate: `0.05`

## Validation
The initial manual calculation matched autograd:
- Loss: `10.5`
- Weight gradient: `-14`
- Bias gradient: `-6`

Three updates produced losses of approximately `10.5 → 2.1167 → 0.4531`, with final parameters `w ≈ 1.6544` and `b ≈ 0.4843`.

## Key Distinction
The gradient measures local sensitivity. Plain SGD applies the parameter change:

$$
\Delta w = -\eta\frac{\partial L}{\partial w}.
$$

Predictions and gradients must be recomputed using the updated parameters.

## Scope
The manual training loop required a variable-selection correction during review. This is a small deterministic optimization exercise.

In [1]:
import torch
import numpy as np

In [2]:
training_inputs = torch.tensor([1.0,2.0,3.0])
targets = torch.tensor([2.0,4.0,6.0])

w = torch.tensor(0.5, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [3]:
y_pred = w*training_inputs + b

loss = torch.mean((y_pred - targets)**2)

loss.backward()

print(f"Autograd Loss:    {loss.item():.4f}")
print(f"Autograd dL/dw:   {w.grad.item():.4f}")
print(f"Autograd dL/db:   {b.grad.item():.4f}")

Autograd Loss:    10.5000
Autograd dL/dw:   -14.0000
Autograd dL/db:   -6.0000


In [4]:
training_inputs = torch.tensor([1.0,2.0,3.0])
targets = torch.tensor([2.0,4.0,6.0])

w_manual = torch.tensor(0.5)
b_manual = torch.tensor(0.0)

In [5]:
def train(w,b, mean, sd):
    for i in range(3):
        y_pred = w*training_inputs + b

        errors = (y_pred - targets)

        loss = torch.mean((errors ** 2))

        gw = torch.sum(errors*training_inputs)*(2/3)
        bw = torch.sum(errors)*(2/3)
        
        # mean = mean + 

        w = w - 0.05*gw
        b = b - 0.05*bw

        print(
        f"Step {i+1} | "
        f"y_pred: {y_pred} | "
        f"errors: {errors} | "
        f"loss: {loss:.4f} | "
        f"gw: {gw:.4f} | "
        f"bw: {bw:.4f} | "
        f"w_new: {w:.4f} | "
        f"b_new: {b:.4f}"
    )

In [6]:
train_data = torch.tensor([1.0,3.0,5.0])
val_data = torch.tensor([7.0,9.0])
test_data = torch.tensor([11.0,13.0])

w_dl = torch.tensor(0.5)
b_dl = torch.tensor(0.0)

In [7]:
mean_train = torch.mean(train_data)
std_train = torch.std(train_data, correction=0)

train_norm = (train_data - mean_train) / std_train
val_norm = (val_data - mean_train) / std_train
test_norm = (test_data - mean_train) / std_train

print(f"Training Mean: {mean_train.item():.4f}")
print(f"Training Population Std: {std_train.item():.4f}\n")
print(f"Normalized Train: {train_norm.tolist()}")
print(f"Normalized Val:   {val_norm.tolist()}")
print(f"Normalized Test:  {test_norm.tolist()}")

Training Mean: 3.0000
Training Population Std: 1.6330

Normalized Train: [-1.2247447967529297, 0.0, 1.2247447967529297]
Normalized Val:   [2.4494895935058594, 3.674234390258789]
Normalized Test:  [4.898979187011719, 6.123723983764648]
